# CTA benchmark analysis

Exploratory analysis of `results_bundle.json` artifacts emitted by
`cta metrics compute` (or, transitively, `cta experiment`).

This notebook is **not** authoritative: the canonical CSV, Markdown, and
LaTeX tables that feed paper submissions are produced by
`crates/cta_reports` and regenerated deterministically from the same
bundles. This notebook is for exploratory cross-run slicing and quick
sanity plots during development.

Usage:

```bash
# From the repo root, after running at least one experiment:
cta experiment --config configs/experiments/pilot_v1.json

# Then:
jupyter notebook notebooks/analysis.ipynb
```

The cells below are pure-Python (standard library plus `matplotlib`) and
run to completion against any set of `runs/**/results_bundle.json`
artifacts in the workspace.

In [ ]:
"""Load every results_bundle.json under ../runs and build a per-system frame."""
from __future__ import annotations

import json
from pathlib import Path

WORKSPACE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = WORKSPACE / "runs"

bundles: list[dict] = []
for bundle_path in sorted(RUNS_DIR.glob("run_*/results_bundle.json")):
    with bundle_path.open("r", encoding="utf-8") as fh:
        bundles.append(json.load(fh))

print(f"loaded {len(bundles)} results_bundle.json artifact(s) from {RUNS_DIR}")
for b in bundles:
    rm = b["run_manifest"]
    p = b["aggregate_metrics"]["primary"]
    print(
        f"  {rm['run_id']:<48}  system={rm['system_id']:<18}"
        f"  elab={p['elaboration_rate']:.3f}  faith={p['semantic_faithfulness_mean']:.3f}"
        f"  cov={p['critical_unit_coverage']:.3f}"
    )


In [ ]:
"""Bar chart of primary metrics per system.

Uses matplotlib if available; otherwise prints a text table.
"""
METRIC_KEYS = [
    "elaboration_rate",
    "semantic_faithfulness_mean",
    "critical_unit_coverage",
    "rust_consistency_rate",
    "vacuity_rate",
    "proof_utility",
]

rows = []
for b in bundles:
    rm = b["run_manifest"]
    p = b["aggregate_metrics"]["primary"]
    rows.append(
        {"system_id": rm["system_id"], "run_id": rm["run_id"], **{k: p[k] for k in METRIC_KEYS}}
    )

try:
    import matplotlib.pyplot as plt
    import numpy as np

    systems = [r["system_id"] for r in rows]
    x = np.arange(len(METRIC_KEYS))
    width = 0.8 / max(1, len(systems))

    fig, ax = plt.subplots(figsize=(10, 4))
    for i, r in enumerate(rows):
        offsets = x + (i - (len(systems) - 1) / 2) * width
        ax.bar(offsets, [r[k] for k in METRIC_KEYS], width, label=r["system_id"])
    ax.set_xticks(x)
    ax.set_xticklabels(METRIC_KEYS, rotation=30, ha="right")
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("metric (unit scalar)")
    ax.set_title("Primary metrics by system")
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()
except ModuleNotFoundError:
    header = f"{'system_id':<20} " + " ".join(f"{k[:12]:>13}" for k in METRIC_KEYS)
    print(header)
    for r in rows:
        print(
            f"{r['system_id']:<20} "
            + " ".join(f"{r[k]:13.3f}" for k in METRIC_KEYS)
        )


In [ ]:
"""Per-instance elaboration + critical-unit coverage heatmap (systems x instances)."""
systems = sorted({b["run_manifest"]["system_id"] for b in bundles})
instances = sorted(
    {r["instance_id"] for b in bundles for r in b.get("instance_results", [])}
)

if systems and instances:
    coverage = [[0.0] * len(instances) for _ in systems]
    for b in bundles:
        sid = b["run_manifest"]["system_id"]
        si = systems.index(sid)
        for r in b.get("instance_results", []):
            if r["instance_id"] in instances:
                ii = instances.index(r["instance_id"])
                total = max(1, r.get("critical_units_total", 0))
                coverage[si][ii] = r.get("critical_units_covered", 0) / total

    try:
        import matplotlib.pyplot as plt
        import numpy as np

        fig, ax = plt.subplots(figsize=(max(6, 0.6 * len(instances)), 1 + 0.4 * len(systems)))
        im = ax.imshow(np.array(coverage), aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
        ax.set_xticks(range(len(instances)))
        ax.set_xticklabels(instances, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(systems)))
        ax.set_yticklabels(systems)
        ax.set_title("Critical-unit coverage (systems x instances)")
        fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
        fig.tight_layout()
        plt.show()
    except ModuleNotFoundError:
        for si, sid in enumerate(systems):
            print(f"{sid:<20} " + " ".join(f"{c:.2f}" for c in coverage[si]))
else:
    print("no (system, instance) pairs found; run an experiment first")
